# 分析问题出在哪一步

只看一个总分，无法决定下一步该改什么。错误分析要回到三份原始材料：用户问题、检索到的资料和最终回答。

本页用几个固定例子说明怎样定位问题，不比较某种改法的效果。

In [1]:
examples = [
    {"name": "原文没有答案", "source_has_answer": False, "retrieved": False, "answer_supported": False},
    {"name": "检索漏掉正确页", "source_has_answer": True, "retrieved": False, "answer_supported": False},
    {"name": "资料正确但回答乱加结论", "source_has_answer": True, "retrieved": True, "answer_supported": False},
    {"name": "回答有资料支持", "source_has_answer": True, "retrieved": True, "answer_supported": True},
]

def locate(item):
    if not item["source_has_answer"]:
        return "资料范围"
    if not item["retrieved"]:
        return "读取、分块或检索"
    if not item["answer_supported"]:
        return "回答"
    return "未发现这三类问题"

for item in examples:
    print(item["name"], "→", locate(item))

原文没有答案 → 资料范围
检索漏掉正确页 → 读取、分块或检索
资料正确但回答乱加结论 → 回答
回答有资料支持 → 未发现这三类问题


分析结果应该指向一个可执行的改动：资料范围不足就补资料或拒答，分块丢失就改分块，正确资料排太后就改检索或排序，回答没有使用资料才改 Prompt 或回答规则。



## 从低分到根因：三步检查和归因矩阵

错误分析不能只看总分。先问资料范围是否包含答案，再问正确资料是否进入上下文，再把回答拆成结论检查原文依据。可用下面的矩阵定位：资料不存在或权限不足属于资料范围；资料存在但被分块/排序/检索漏掉属于读取或检索；资料已到位但结论无依据、引用错或漏答属于生成；延迟、超时、路由权限和费用异常属于系统。

每条低分记录都应该留下证据：问题、资料是否有答案、返回页和排名、相关片段、回答声明、支持原文、错误根因和下一步动作。错误归因不是为了给方法贴标签，而是为了决定改资料、分块、检索、Prompt 还是系统配置。


In [2]:
ROOT_CAUSE_ACTIONS = {
    "资料范围": "补充资料、明确拒答范围或记录权限限制",
    "读取/分块": "检查解析、页边界、父子关系和字符截断",
    "检索": "改检索问题、过滤条件、排序或返回数量，并保留改动前参照",
    "生成": "逐条核对可核对的结论，修改提示词、引用和资料不足规则",
    "系统": "检查路由、超时、重试、成本和脱敏日志",
}

def attribute_error(source_has_answer, retrieved, claims_supported, system_ok=True):
    if not system_ok: return "系统"
    if not source_has_answer: return "资料范围"
    if not retrieved: return "检索"
    if not claims_supported: return "生成"
    return "未发现明显问题"

def root_cause_record(case_id, question, expected_pages, returned_pages, claims):
    found = bool(set(expected_pages) & set(returned_pages))
    supported = all(bool(claim.get("supported")) for claim in claims) if claims else False
    cause = attribute_error(True, found, supported)
    return {"case_id": case_id, "question": question, "expected_pages": expected_pages,
            "returned_pages": returned_pages, "root_cause": cause,
            "next_action": ROOT_CAUSE_ACTIONS.get(cause, "人工复核")}

# 只有保存了 query/context/answer 三份原始材料，归因结果才可复查。
example_root_record = root_cause_record(
    "lda_recursive_derivation",
    "LDA 从投影分离目标怎样推到广义特征值？",
    [41, 42, 43, 44], [44, 131, 126, 139],
    [{"supported": False}],
)
print(
    f"错误归因示例：初步归因是‘{example_root_record['root_cause']}’；"
    f"下一步：{example_root_record['next_action']}。"
)

错误归因示例：初步归因是‘生成’；下一步：逐条核对可核对的结论，修改提示词、引用和资料不足规则。


## 从错误分析到修复验证

一个低分记录至少要保存问题、资料是否包含答案、返回页和排名、实际上下文、回答中的可核对结论、原文证据、根因和下一步动作。修复时只改一个主要变量，仍用同一问题、同一份资料版本和同一套评分规则重跑；修复后除了看目标问题，还要检查一个原本正常的问题有没有退化。

| 观察到的证据 | 先归到哪里 | 可以做的修复 |
| --- | --- | --- |
| 原资料就没有答案，或权限不允许读 | 资料范围 | 补资料、明确范围，或按规则拒答 |
| 有答案但解析/分块破坏了证据 | 读取/分块 | 修复解析、页边界、父子关系和截断 |
| 资料存在但没被返回或排得太后 | 检索 | 改检索问题、过滤、返回数量或排序，保留改动前参照 |
| 资料已到位但结论无依据、漏答或引用错 | 生成 | 改回答规则、引用格式和资料不足时的行为 |
| 超时、权限、路由、成本异常 | 系统 | 修复配置并单独监控，不能把它算成回答质量提升 |

In [3]:
def triage_and_repair(record, *, source_has_answer, context_complete, claims_supported, system_ok=True):
    """从证据决定修复方向，并返回需要保留的复查键。"""
    if not system_ok:
        cause = "系统"
    elif not source_has_answer:
        cause = "资料范围"
    elif not context_complete:
        cause = "读取/分块"
    elif not record.get("retrieved", False):
        cause = "检索"
    elif not claims_supported:
        cause = "生成"
    else:
        cause = "未发现明显问题"
    return {
        "case_id": record["case_id"],
        "root_cause": cause,
        "next_action": ROOT_CAUSE_ACTIONS.get(cause, "人工复核"),
        "rerun_key": {"case_id": record["case_id"], "dataset_version": record.get("dataset_version"),
                       "index_version": record.get("index_version")},
    }

# 归因结论也要标出证据是否完整；不要把规则输出称为自动证明。
example_record = {"case_id": "lda_recursive_derivation", "retrieved": False,
                  "dataset_version": "snapshot", "index_version": "bge-saved"}
example_plan = triage_and_repair(example_record, source_has_answer=True,
                               context_complete=True, claims_supported=False)
assert example_plan["root_cause"] == "检索"
assert example_plan["rerun_key"]["case_id"] == example_record["case_id"]
print(
    f"复跑计划示例：先保留改动前结果，再调整检索问题、过滤条件、排序或返回数量；"
    "数据使用固定版本，向量库使用教程随附的 BGE 向量库。"
)

复跑计划示例：先保留改动前结果，再调整检索问题、过滤条件、排序或返回数量；数据使用固定版本，向量库使用教程随附的 BGE 向量库。


## 本页导航

- 章节入口：[本章 README](README.md)
- 运行准备：[C7 统一运行准备](../README.md#运行准备)
- 相关下一步：[比较改动前后](比较改动前后.ipynb)

